# Sentinel-1 float + Gamma MAP smoke notebook

This is the dedicated, durable smoke notebook for the
`sentinel1_float` workflow with opt-in mono-temporal Gamma MAP
speckle filtering (`gamma_map()`).

What this notebook covers:

1. Earth Engine initialization
2. AOI normalization
3. Unfiltered `sentinel1_float` linear-unit composite (baseline)
4. `sentinel1_float` + `gamma_map()` filtered composite
5. `sentinel1_float` + `gamma_map()` + VH/VV ratio expression transform
6. Optional Drive export (gated)
7. A clean run summary

## When to use which Sentinel-1 preset

| Preset             | Collection              | Units    | Use for                                           |
|--------------------|-------------------------|----------|---------------------------------------------------|
| `sentinel1`        | `COPERNICUS/S1_GRD`     | dB       | Direct VV/VH band composites; visual inspection.  |
| `sentinel1_float`  | `COPERNICUS/S1_GRD_FLOAT` | linear | Ratio / algebraic SAR features (VH/VV, RVI, etc.). |

## Why `gamma_map()` is opt-in

Gamma MAP filtering is a meaningful but expensive preprocessing step
that changes pixel values. Making it opt-in keeps unit semantics
explicit, preserves the unfiltered path, and fits the function-based
`compose(..., preprocess=...)` slot without hidden defaults.

## What this still does *not* provide

`geecomposer` is not a Sentinel-1 ARD framework. The following are
intentionally out of scope:

- multi-temporal speckle filtering
- Refined Lee, Lee Sigma, Boxcar, or any user-facing filter menu
- radiometric terrain flattening / slope correction
- additional border-noise correction beyond the EE ingestion default
- tide-aware filtering
- SAR texture features

Apply `gamma_map()` only to `sentinel1_float`. Applying it to dB-scaled
imagery silently produces wrong values.

## Live-validation assumptions

Same as the end-to-end smoke notebook: `authenticate=False` by default,
hardcoded `project="manglariars"`, exports gated behind
`START_EXPORT = False`.


## 1. Path setup and imports


In [ ]:
from pathlib import Path
import sys

import ee


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "geecomposer_v0.1_spec.md").exists() and (
            candidate / "08_pkg" / "src"
        ).exists():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory.")


repo_root = find_repo_root(Path.cwd().resolve())
pkg_src = repo_root / "08_pkg" / "src"
aoi_path = repo_root / "01_data" / "case_studies" / "rbmn.geojson"

if str(pkg_src) not in sys.path:
    sys.path.insert(0, str(pkg_src))

from geecomposer import compose, export_to_drive, initialize
from geecomposer.datasets.sentinel1_preprocessing import gamma_map
from geecomposer.transforms.expressions import expression_transform

print("Repo root:    ", repo_root)
print("AOI path:     ", aoi_path)


## 2. Initialize Earth Engine


In [ ]:
initialize(project="manglariars", authenticate=False)
assert ee.Number(1).getInfo() == 1
print("Earth Engine initialized.")


## 3. Baseline: unfiltered Sentinel-1 float VV median

This is the reference unfiltered linear-unit composite. Use it as a
side-by-side comparison for the filtered version below.


In [ ]:
s1f_vv_unfiltered = compose(
    dataset="sentinel1_float",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    select="VV",
    reducer="median",
    filters={
        "instrumentMode": "IW",
        "polarizations": ["VV"],
        "orbitPass": "ASCENDING",
    },
)

unfiltered_bands = s1f_vv_unfiltered.bandNames().getInfo()
unfiltered_props = s1f_vv_unfiltered.getInfo()["properties"]

assert unfiltered_bands == ["VV"]
assert unfiltered_props["geecomposer:dataset"] == "sentinel1_float"
assert unfiltered_props["geecomposer:collection"] == "COPERNICUS/S1_GRD_FLOAT"

{"bands": unfiltered_bands, "properties": unfiltered_props}


## 4. Filtered: Sentinel-1 float VV median + Gamma MAP

Same query, but each image is passed through `gamma_map()` before the
temporal reduction. Default `kernel_size=7`; ENL is fixed internally
to 5.

Note that metadata still records `geecomposer:transform` as empty,
because `gamma_map()` is a *preprocess* (per-image preparation), not
a transform (per-image feature). This separation is intentional.


In [ ]:
s1f_vv_filtered = compose(
    dataset="sentinel1_float",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    select="VV",
    preprocess=gamma_map(),
    reducer="median",
    filters={
        "instrumentMode": "IW",
        "polarizations": ["VV"],
        "orbitPass": "ASCENDING",
    },
)

filtered_bands = s1f_vv_filtered.bandNames().getInfo()
filtered_props = s1f_vv_filtered.getInfo()["properties"]

assert filtered_bands == ["VV"]
assert filtered_props["geecomposer:dataset"] == "sentinel1_float"

{"bands": filtered_bands, "properties": filtered_props}


## 5. Filtered + transform: VH/VV ratio in linear units

This is the canonical biomass-style SAR feature workflow:

1. load `sentinel1_float` (linear units required for ratios)
2. preprocess with `gamma_map()` (speckle reduction)
3. transform per image with VH/VV expression
4. reduce across time with `median`

The ratio is computed in linear power, **not** in dB. dB ratios are
not physically meaningful.


In [ ]:
vh_vv = expression_transform(
    expression="vh / vv",
    band_map={"vh": "VH", "vv": "VV"},
    name="vh_vv_ratio",
)

s1f_ratio = compose(
    dataset="sentinel1_float",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    preprocess=gamma_map(),
    transform=vh_vv,
    reducer="median",
    filters={
        "instrumentMode": "IW",
        "polarizations": ["VV", "VH"],
    },
)

ratio_bands = s1f_ratio.bandNames().getInfo()
ratio_props = s1f_ratio.getInfo()["properties"]

assert ratio_bands == ["vh_vv_ratio"]
assert ratio_props["geecomposer:transform"] == "expression_transform('vh_vv_ratio')"

{"bands": ratio_bands, "properties": ratio_props}


## 6. Optional Drive export (gated)

Same explicit start gate as in the end-to-end notebook. Flip
`START_EXPORT = True` only when you actually want to export.


In [ ]:
drive_folder = "geecomposer-dev"
export_description = "rbmn_s1f_vh_vv_ratio_gammamap_2025_smoke"

ratio_task = export_to_drive(
    image=s1f_ratio,
    description=export_description,
    folder=drive_folder,
    region=str(aoi_path),
    scale=10,
)

START_EXPORT = False

if START_EXPORT:
    ratio_task.start()
    print("Export task started.")
else:
    print("Skipping start. Set START_EXPORT = True to launch.")

ratio_task.status()


## 7. Run summary


In [ ]:
summary = {
    "aoi_path": str(aoi_path),
    "unfiltered_bands": unfiltered_bands,
    "filtered_bands": filtered_bands,
    "ratio_bands": ratio_bands,
    "ratio_transform_meta": ratio_props["geecomposer:transform"],
    "export_description": export_description,
    "export_started": START_EXPORT,
    "export_state": ratio_task.status().get("state"),
}
summary
